# PCAG Research — 무료 Colab(T4) 실측 실행

클릭 한 번으로 `Source=Measured-GPU` 를 추출하고 17개 차트를 갱신합니다.

**실행 순서**: (1) 런타임 T4 설정 → (2) Drive 마운트 → (3) 설치 → (4) 모델 셀 실행.

세션이 끊기면 노트북을 다시 열어 **Run all** → `--resume` 이 진행분을 이어받습니다.


## 0. GPU 확인
`런타임 > 런타임 유형 변경 > T4 GPU` 후 아래를 실행해 확인하세요.

In [ ]:
import torch, subprocess
print('cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(torch.cuda.get_device_name(0), f'{p.total_memory/1024**3:.1f} GB')
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)


## 1. Drive 마운트 + 저장소 클론
인증 팝업을 허용하세요. **REPO 주소를 본인 저장소로 바꾸세요.**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/pcag_results', exist_ok=True)
print('Drive OK')

REPO = 'https://github.com/<your-org>/llm-pcag-research.git'  # ← 여기를 수정
%cd /content
if not os.path.exists('/content/llm-pcag-research'):
    !git clone {REPO}
%cd /content/llm-pcag-research
!git pull --ff-only 2>/dev/null || true
print('repo ready')


## 2. 패키지 설치
(Colab 기본 torch 는 유지 — transformers/accelerate/bnb 최신판만 설치)

In [ ]:
!pip install -q --upgrade transformers accelerate bitsandbytes pynvml
!pip install -q datasets scipy sympy matplotlib

import torch, transformers, datasets
print('torch', torch.__version__, '| transformers', transformers.__version__)
print('datasets', datasets.__version__)


## 3. (선택) HF 토큰
`meta-llama/Llama-3-8B`, `google/gemma-2-9b` 는 라이선스 동의가 필요합니다.
공개 모델(Qwen/Mistral)만 실측하면 빈 값으로 Enter 해도 됩니다.

In [ ]:
import getpass, os
if not os.environ.get('HF_TOKEN'):
    tok = getpass.getpass('HF read token (공개 모델만이면 Enter): ')
    if tok:
        os.environ['HF_TOKEN'] = tok
print('HF_TOKEN set:', bool(os.environ.get('HF_TOKEN')))


## 4. 실측 실행 — 모델별 셀
**세션당 1셀** 실행을 권장합니다 (Colab 타임아웃). 세션이 끊기면 노트북 재실행 → `--resume` 으로 이어집니다.

소요: 모델 1개(7정밀도 × 3반복 + ARC-Easy 100문항) ≈ 35–50분. `--repeats 1` 로 줄이면 15–20분.

### 4. Llama-3-8B (기준 모델 + results_raw.csv 갱신)

In [ ]:
%cd /content/llm-pcag-research/experiments
!python benchmark_driver.py \
    --models meta-llama/Llama-3-8B \
    --quant-method rtn \
    --precisions FP16 INT8 INT6 INT5 INT4 INT3 INT2 \
    --eval arc_easy --eval_questions 100 \
    --repeats 3 --energy_norm --update-main \
    --drive-dir /content/drive/MyDrive/pcag_results \
    --resume


### 4. Qwen-2.5-7B

In [ ]:
%cd /content/llm-pcag-research/experiments
!python benchmark_driver.py \
    --models Qwen/Qwen2.5-7B \
    --quant-method rtn \
    --precisions FP16 INT8 INT6 INT5 INT4 INT3 INT2 \
    --eval arc_easy --eval_questions 100 \
    --repeats 3 --energy_norm \
    --drive-dir /content/drive/MyDrive/pcag_results \
    --resume


### 4. Gemma-2-9B (HF 토큰 필요)

In [ ]:
%cd /content/llm-pcag-research/experiments
!python benchmark_driver.py \
    --models google/gemma-2-9b \
    --quant-method rtn \
    --precisions FP16 INT8 INT6 INT5 INT4 INT3 INT2 \
    --eval arc_easy --eval_questions 100 \
    --repeats 3 --energy_norm \
    --drive-dir /content/drive/MyDrive/pcag_results \
    --resume


### 4. Mistral-7B

In [ ]:
%cd /content/llm-pcag-research/experiments
!python benchmark_driver.py \
    --models mistralai/Mistral-7B \
    --quant-method rtn \
    --precisions FP16 INT8 INT6 INT5 INT4 INT3 INT2 \
    --eval arc_easy --eval_questions 100 \
    --repeats 3 --energy_norm \
    --drive-dir /content/drive/MyDrive/pcag_results \
    --resume


## 5. (선택) bnb 표준 방법 검증 세트
균일 RTN 곡선의 경향이 표준 양자화와 질적으로 일치하는지 별도 CSV로 확인.

In [ ]:
%cd /content/llm-pcag-research/experiments
!python benchmark_driver.py \
    --models meta-llama/Llama-3-8B \
    --quant-method bnb --precisions INT8 INT4 \
    --eval arc_easy --eval_questions 100 --repeats 3 --energy_norm \
    --drive-dir /content/drive/MyDrive/pcag_results \
    --resume


## 6. 실측 후 파이프라인 — 17개 차트 갱신
Drive 에 저장된 결과를 복원 후 분석 파이프라인을 순서대로 실행합니다.

In [ ]:
%cd /content/llm-pcag-research/experiments
!cp /content/drive/MyDrive/pcag_results/results_raw.csv . 2>/dev/null || true
!cp /content/drive/MyDrive/pcag_results/results_multimodel_raw.csv . 2>/dev/null || true
!cp /content/drive/MyDrive/pcag_results/results_multimodel_bnb_raw.csv . 2>/dev/null || true

!python analysis.py
!python analytical_proof.py
!python sensitivity.py
!python jevons_model.py
!python make_figures.py
!python dry_run.py


## 7. 결과 확인
- 원시 데이터: `experiments/results_raw.csv`, `results_multimodel_raw.csv`
- 분석: `analysis_summary.json`, `analysis_proof.json`, `sensitivity_summary.json`
- 그림 17종: `docs/figures/fig*.png` + `fig*.pdf`


In [ ]:
%cd /content/llm-pcag-research/experiments
import os, csv, glob
for f in ['results_raw.csv', 'results_multimodel_raw.csv']:
    if os.path.exists(f):
        print('='*20, f, '='*20)
        with open(f) as fh:
            print(fh.read())
figs = sorted(glob.glob('/content/llm-pcag-research/docs/figures/*.png'))
print('figures:', len(figs))

# (선택) 결과를 로컬로 다운로드
from google.colab import files
files.download('results_multimodel_raw.csv')  # 원하면 주석 해제
